In [17]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_postgres import PGVector
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

In [18]:
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

In [19]:
#########################################
# 1. Load PDF
#########################################

loader = PyPDFLoader("1992_ghana_constitution.pdf")
documents = loader.load()

print(f"Loaded {len(documents)} pages")

Loaded 47 pages


In [20]:
#########################################
# 2. Split into chunks
#########################################

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

chunks = splitter.split_documents(documents)


print(chunks[:5])

print(f"Chunks: {len(chunks)}")

[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20111118145154', 'title': 'THE CONSTITUTION OF THE REPUBLIC OF GHANA', 'author': 'Talk', 'moddate': '2016-12-21T16:07:42-08:00', 'source': '1992_ghana_constitution.pdf', 'total_pages': 47, 'page': 0, 'page_label': '1'}, page_content='THE CONSTITUTION OF THE REPUBLIC OF GHANA \nCHAPTER ONE \n1 (1) The Sovereignty of Ghana resides in the people of Ghana in whose name and for whose welfare the \npowers of government are to be exercised in the manner and within the limits laid down in this Constitution.'), Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20111118145154', 'title': 'THE CONSTITUTION OF THE REPUBLIC OF GHANA', 'author': 'Talk', 'moddate': '2016-12-21T16:07:42-08:00', 'source': '1992_ghana_constitution.pdf', 'total_pages': 47, 'page': 0, 'page_label': '1'}, page_content='(2) This Co

In [21]:
#########################################
# 3. Create embeddings
#########################################

# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [22]:
#################################################
# 4. PGVector
#################################################

# Add the document chunks to the "vector store" using OpenAIEmbeddings
# vectorstore = InMemoryVectorStore.from_documents(
#     documents=chunks,
#     embedding=OpenAIEmbeddings(),
# )

vectorstore = PGVector(
    embeddings=embeddings,
    collection_name="constitution_handbook",
    connection=DATABASE_URL,
    use_jsonb=True,
)

In [23]:
#################################################
# 5. Insert documents
#################################################

vectorstore.add_documents(chunks)

print("Documents stored!")

Documents stored!


In [24]:
#########################################
# 6. Create Retriever
#########################################
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5})

print(retriever)

tags=['PGVector', 'OpenAIEmbeddings'] vectorstore=<langchain_postgres.vectorstores.PGVector object at 0x114f68190> search_type='mmr' search_kwargs={'k': 5}


### Important: Score Meaning Depends on Distance Strategy
- With pgvector, the score returned may be a distance, not cosine similarity.

- 0 = identical vectors, 
- 1 = completely different

so 
- 0.05  -> very similar
- 0.70  -> weak match
- cosine_similarity = 1 - cosine_distance

In [25]:
docs_with_scores = vectorstore.similarity_search_with_score(
    query="How to become a citizen of Ghana?",
    k=3
)

for doc, score in docs_with_scores:
    print("Score:", score)
    print("Content:")
    print(doc.page_content)
    print("-" * 50)

Score: 0.3431955436520465
Content:
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
--------------------------------------------------
Score: 0.3431955436520465
Content:
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
--------------------------------------------------
Score: 0.3432332318046809
Content:
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
----------------------

In [26]:
for doc, distance in docs_with_scores:
    similarity = 1 - distance
    print(f"Similarity: {similarity:.3f}")
    print(doc.page_content)

Similarity: 0.657
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
Similarity: 0.657
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
Similarity: 0.657
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.


## In production

In [27]:
query = "how to become a citizen of ghana?"

results = vectorstore.similarity_search_with_score(query, k=5)

for doc, distance in results:
    similarity = 1 - distance
    if similarity > 0.60:
        print("Use this document:")
        print(doc.page_content)
    else:
        print("Low confidence - ignore")

Use this document:
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
Use this document:
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
Use this document:
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
Use this document:
8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a 

In [28]:
#########################################
# 7. Ask a Question
#########################################

question = "how to become a citizen of ghana?"

docs = retriever.invoke(question)

print("\nRetrieved Documents\n")

for doc in docs:
    print(doc.page_content)
    print("-" * 50)


Retrieved Documents

8 (1) Subject to this article, a citizen of Ghana shall cease forthwith to be a citizen of Ghana if, on attaining 
the age of twenty-one years, he, by a voluntary act, other than marriage, acquires or retains the citizenship 
of a country other than Ghana.
--------------------------------------------------
citizenship of a country other than Ghana shall, on the renunciation of his citizenship of that other country, 
become a citizen of Ghana. 
4) Where the law of a country, other than Ghana, requires a person who marries a citizen of that country to
--------------------------------------------------
9 (1) Parliament may make provision for the acquisition of citizenship of Ghana by persons who are not 
eligible to become citizens of Ghana under the provisions of this Constitution. 
(2) Except as otherwise provided in article & of this Constitution, a person shall not be registered as a citizen
--------------------------------------------------
presumed to be a citi

In [29]:
#########################################
# 8. Send Context to LLM
#########################################
from IPython.display import Markdown, display

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
Answer ONLY from the provided context.

Context:
{context}

Question:
{question}
"""

llm = ChatOpenAI(model="gpt-4o-mini")

response = llm.invoke(prompt)

print("\nAnswer\n")
display(Markdown(response.content))


Answer



To become a citizen of Ghana, one can acquire citizenship through the following means:

1. If an individual is adopted by a citizen of Ghana, they become a citizen of Ghana by virtue of the adoption, provided they are not more than sixteen years old and neither of their parents is a citizen of Ghana.

2. Parliament may make provisions for the acquisition of citizenship of Ghana by people who are not eligible under the provisions of the Constitution.

Additionally, a person who renounces their citizenship of another country and does so in accordance with the law of that country may become a citizen of Ghana.

In [32]:
#########################################
# 8. Send Context to LLM
#########################################
from IPython.display import Markdown, display

question = "Parliament may by law provide for the delimitation"

docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
Answer ONLY from the provided context.

Context:
{context}

Question:
{question}
"""

llm = ChatOpenAI(model="gpt-4o-mini")

response = llm.invoke(prompt)

print("\nAnswer\n")
display(Markdown(response.content))


Answer



of the territorial sea, the contiguous zone, the exclusive economic zone and the continental shelf of Ghana.